#  LangChain Full Application
## Prompt Template → Groq Model → Output Parser → Chain

###  What We Build:
A **Data Science Mentor App** that answers any Data Science question using a full LangChain pipeline.

---
##  Step 1: Install Libraries

In [ ]:
!pip install langchain
!pip install langchain-groq
!pip install langchain-core

---
##  Step 2: Load Groq API Key

In [2]:
# Load API key from .txt file
with open("groq_api_key.txt", "r") as f:
    api_key = f.read().strip()

print(" API Key loaded!")

 API Key loaded!


---
##  COMPONENT 1 — Prompt Template

A **Prompt Template** lets you define a reusable prompt with placeholders.  
Here we set a **system prompt** to make the model act as a Data Science Mentor.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

# Define the Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert Data Science Mentor with 10+ years of experience.
    Your job is to teach Data Science concepts clearly and simply.
    - Always give a clear explanation
    - Use a real-world example
    - End with one pro tip
    """),
    ("human", "{question}")
])

#  Preview the template
print(" Prompt Template Created!")
print()
print(prompt)

 Prompt Template Created!

input_variables=['question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n    You are an expert Data Science Mentor with 10+ years of experience.\n    Your job is to teach Data Science concepts clearly and simply.\n    - Always give a clear explanation\n    - Use a real-world example\n    - End with one pro tip\n    '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [4]:
# See how the prompt looks when filled with a question
sample_prompt = prompt.invoke({"question": "What is overfitting?"})

print(" Filled Prompt Preview:")
print("-" * 40)
for msg in sample_prompt.messages:
    print(f"[{msg.type.upper()}]: {msg.content}")

 Filled Prompt Preview:
----------------------------------------
[SYSTEM]: 
    You are an expert Data Science Mentor with 10+ years of experience.
    Your job is to teach Data Science concepts clearly and simply.
    - Always give a clear explanation
    - Use a real-world example
    - End with one pro tip
    
[HUMAN]: What is overfitting?


---
##  COMPONENT 2 — Groq Model (LLaMA 3)

The **Model** is where the actual AI processing happens.  
We use **Groq** to run the open-source **LLaMA 3** model for free.

In [5]:
from langchain_groq import ChatGroq

# Initialize the Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=api_key,
    temperature=0.7
)

#  Preview the model config
print(" Groq Model Initialized!")
print()
print(f"  Model Name  : {model.model_name}")
print(f"  Temperature : {model.temperature}")

 Groq Model Initialized!

  Model Name  : llama-3.3-70b-versatile
  Temperature : 0.7


In [6]:
# Test the model alone (without chain)
from langchain_core.messages import HumanMessage, SystemMessage

test_response = model.invoke([
    SystemMessage(content="You are a Data Science Mentor."),
    HumanMessage(content="What is a neural network in one line?")
])

print(" Model standalone test:")
print(test_response.content)

 Model standalone test:
A neural network is a machine learning model inspired by the human brain's structure and function, composed of interconnected nodes (neurons) that process and transmit information to make predictions or decisions.


---
## COMPONENT 3 — Output Parser

The **Output Parser** transforms the raw model response into a clean, usable format.  
Without it, you get an `AIMessage` object. With it, you get a plain **string**.

In [7]:
from langchain_core.output_parsers import StrOutputParser

# Initialize the Output Parser
output_parser = StrOutputParser()

#  Show what it does
raw_response = model.invoke("Say hello in one word")

print(" WITHOUT Output Parser (raw AIMessage object):")
print(type(raw_response))
print(raw_response)

print()

parsed_response = output_parser.invoke(raw_response)
print(" WITH Output Parser (clean string):")
print(type(parsed_response))
print(parsed_response)

 WITHOUT Output Parser (raw AIMessage object):
<class 'langchain_core.messages.ai.AIMessage'>
content='Hello' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 40, 'total_tokens': 42, 'completion_time': 0.006724673, 'completion_tokens_details': None, 'prompt_time': 0.003227069, 'prompt_tokens_details': None, 'queue_time': 0.162023355, 'total_time': 0.009951742}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ae07559190', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d8b9f-85d3-76d1-9c54-43d87db87025-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 2, 'total_tokens': 42}

 WITH Output Parser (clean string):
<class 'langchain_core.messages.base.TextAccessor'>
Hello


---
##  Step 3: Create the Chain

Now we **connect all 3 components** using the `|` pipe operator (LCEL — LangChain Expression Language).

```
Prompt Template  →  Groq Model  →  Output Parser
```

In [8]:
# 🔗 Build the Chain using LCEL pipe operator
chain = prompt | model | output_parser

print(" Chain Created!")
print()
print("Pipeline:  Prompt Template  |  Groq LLaMA 3  |  StrOutputParser")

 Chain Created!

Pipeline:  Prompt Template  |  Groq LLaMA 3  |  StrOutputParser


---
## 🚀 Step 4: Run the Full Application

In [9]:
#  Ask your first question to the Data Science Mentor!
response = chain.invoke({"question": "What is overfitting in machine learning?"})

print(" Data Science Mentor says:")
print("=" * 50)
print(response)

 Data Science Mentor says:
Overfitting is a fundamental concept in machine learning that occurs when a model is too complex and learns the noise in the training data, rather than the underlying patterns. As a result, the model performs exceptionally well on the training data but poorly on new, unseen data.

Let me illustrate this with a real-world example. Imagine you're trying to build a model to predict house prices based on features like the number of bedrooms, square footage, and location. You collect a dataset of 100 houses and train a model on it. The model is so complex that it learns the exact prices of each house in the training dataset, including any errors or outliers. When you test the model on the training data, it predicts the prices with 100% accuracy. However, when you try to use the model to predict the price of a new house that wasn't in the training dataset, it performs poorly.

This is because the model has overfit the training data, learning the noise and random fl

In [10]:
response2 = chain.invoke({"question": "Explain the difference between supervised and unsupervised learning."})

print(" Data Science Mentor says:")
print("=" * 50)
print(response2)

 Data Science Mentor says:
In Data Science, machine learning is a crucial concept that enables computers to learn from data. There are two primary types of machine learning: supervised and unsupervised learning.

**Supervised Learning:**
Supervised learning is a type of machine learning where the computer is trained on labeled data. This means that the data is already categorized or classified, and the computer learns to predict the correct label or category for new, unseen data. Think of it like teaching a child to recognize different types of animals. You show them pictures of cats and dogs, and tell them which one is which. The child learns to associate the characteristics of each animal with its corresponding label.

For example, let's say we want to build a system that can predict whether a person is likely to buy a car based on their age, income, and location. We collect data on these factors and label each person as either "buyer" or "non-buyer". We then use this labeled data to

In [11]:
response3 = chain.invoke({"question": "What is the bias-variance tradeoff?"})

print(" Data Science Mentor says:")
print("=" * 50)
print(response3)

 Data Science Mentor says:
The bias-variance tradeoff is a fundamental concept in machine learning and data science. It refers to the inherent tradeoff between two types of errors that can occur when training a model: bias and variance.

**Bias** refers to the error introduced by simplifying assumptions or approximations in the model. In other words, it's the difference between the model's expected prediction and the true value. A model with high bias is one that is overly simplistic and fails to capture the underlying patterns in the data.

**Variance**, on the other hand, refers to the error introduced by the noise or randomness in the data. A model with high variance is one that is overly complex and fits the noise in the training data, rather than the underlying patterns.

To illustrate this concept, let's consider a real-world example. Suppose we're trying to predict the price of a house based on its features, such as the number of bedrooms, square footage, and location. We collec

---
##  Step 5: Interactive Application

Ask **any** Data Science question and get a mentored answer!

In [ ]:
#  Interactive Data Science Mentor App
print(" Welcome to the Data Science Mentor App!")
print("Type 'exit' to quit.\n")

while True:
    user_question = input(" Ask a Data Science question: ")
    
    if user_question.lower() == "exit":
        print(" Goodbye! Keep learning Data Science!")
        break
    
    if user_question.strip() == "":
        print(" Please enter a question.\n")
        continue
    
    print("\n Mentor is thinking...\n")
    answer = chain.invoke({"question": user_question})
    
    print("=" * 50)
    print(answer)
    print("=" * 50)
    print()

---
##  Full Summary

```
┌─────────────────────┐
│   COMPONENT 1       │
│   Prompt Template   │  ← System prompt (Data Science Mentor)
│   + {question}      │    + user's question as variable
└────────┬────────────┘
         │  (pipe)
┌────────▼────────────┐
│   COMPONENT 2       │
│   Groq Model        │  ← LLaMA 3.3 70B (free, fast)
│   LLaMA 3.3 70B     │
└────────┬────────────┘
         │  (pipe)
┌────────▼────────────┐
│   COMPONENT 3       │
│   Output Parser     │  ← Converts AIMessage → clean string
│   StrOutputParser   │
└─────────────────────┘
```

| Component | Class Used | Purpose |
|-----------|------------|---------|
| Prompt Template | `ChatPromptTemplate` | Structure the input with system + user message |
| Model | `ChatGroq` | Run LLaMA 3 via Groq API |
| Output Parser | `StrOutputParser` | Extract clean text from response |
| Chain | `prompt \| model \| parser` | Connect all components with LCEL |

###  Useful Links
- [LangChain LCEL Docs](https://python.langchain.com/docs/expression_language/)
- [Groq Console](https://console.groq.com)
- [LangChain Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/)